In [1]:
"""
Примеры кода к Лекции 5.9: Functional API — агенты без графа

Этот модуль демонстрирует:
1. Базовый пайплайн — @entrypoint и @task
2. Условная логика и циклы — обычный Python
3. Побочные эффекты и детерминизм (gotcha)
4. Retry и кэширование на уровне задачи
5. Параллельное выполнение задач
6. entrypoint.final — разделение возвращаемого и сохраняемого
7. previous — короткосрочная память между вызовами
8. interrupt() в Functional API
9. Кастомный стриминг через get_stream_writer()
"""

'\nПримеры кода к Лекции 5.9: Functional API — агенты без графа\n\nЭтот модуль демонстрирует:\n1. Базовый пайплайн — @entrypoint и @task\n2. Условная логика и циклы — обычный Python\n3. Побочные эффекты и детерминизм (gotcha)\n4. Retry и кэширование на уровне задачи\n5. Параллельное выполнение задач\n6. entrypoint.final — разделение возвращаемого и сохраняемого\n7. previous — короткосрочная память между вызовами\n8. interrupt() в Functional API\n9. Кастомный стриминг через get_stream_writer()\n'

In [2]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langgraph.types import CachePolicy, Command, RetryPolicy, interrupt

from llm_config import check_api_key, get_llm

In [3]:
if not check_api_key():
    raise ValueError("API key is not set")

In [4]:
# ============================================================================
# ЧАСТЬ 1: БАЗОВЫЙ ПАЙПЛАЙН
# ============================================================================
def example_basic_pipeline():
    """
    Простейший пайплайн: research → write → edit.
    Каждый @task автоматически создаёт checkpoint.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 1: Базовый пайплайн")
    print("=" * 60)

    llm = get_llm()

    @task
    def research(topic: str) -> str:
        """Исследование темы."""
        msg = llm.invoke(f"Перечисли 3 ключевых факта о: {topic}. Кратко.")
        return msg.content

    @task
    def write_article(topic: str, facts: str) -> str:
        """Написание статьи на основе фактов."""
        msg = llm.invoke(
            f"Напиши короткую статью о '{topic}' на основе фактов:\n{facts}"
        )
        return msg.content

    @task
    def edit_article(draft: str) -> str:
        """Редактирование статьи."""
        msg = llm.invoke(f"Отредактируй и улучши эту статью:\n{draft}")
        return msg.content

    @entrypoint(checkpointer=InMemorySaver())
    def article_pipeline(topic: str) -> str:
        facts = research(topic).result()
        draft = write_article(topic, facts).result()
        final = edit_article(draft).result()
        return final

    config = {"configurable": {"thread_id": "article-1"}}
    result = article_pipeline.invoke("LangGraph", config)
    print(f"  Статья: {result[:200]}...")


if __name__ == "__main__":
    example_basic_pipeline()


ПРИМЕР 1: Базовый пайплайн


  Статья: Конечно — вот отредактированная и более гладкая версия статьи:

**LangGraph** — это фреймворк от LangChain, предназначенный для создания LLM-агентов и приложений в виде **графов состояний**. Такой под...


In [5]:
# ============================================================================
# ЧАСТЬ 2: УСЛОВНАЯ ЛОГИКА И ЦИКЛЫ
# ============================================================================
def example_conditions_and_loops():
    """
    Цикл рефлексии: генерируй → оценивай → улучшай.
    Обычный for + if + break — никаких conditional edges.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 2: Условная логика и циклы")
    print("=" * 60)

    llm = get_llm()

    @task
    def generate(topic: str) -> str:
        msg = llm.invoke(f"Напиши короткую шутку про {topic}.")
        return msg.content

    @task
    def evaluate(joke: str) -> str:
        msg = llm.invoke(
            f"Оцени эту шутку. Если хорошая — ответь APPROVED. "
            f"Если плохая — напиши критику.\nШутка: {joke}"
        )
        return msg.content

    @task
    def improve(joke: str, critique: str) -> str:
        msg = llm.invoke(
            f"Улучши шутку на основе критики.\n"
            f"Шутка: {joke}\nКритика: {critique}"
        )
        return msg.content

    @entrypoint(checkpointer=InMemorySaver())
    def joke_pipeline(topic: str) -> str:
        joke = generate(topic).result()
        print(f"  Черновик: {joke[:80]}...")

        for i in range(3):
            critique = evaluate(joke).result()
            print(f"  Итерация {i + 1}: {critique[:60]}...")
            if "APPROVED" in critique.upper():
                print("  Одобрено!")
                break
            joke = improve(joke, critique).result()

        return joke

    config = {"configurable": {"thread_id": "joke-1"}}
    result = joke_pipeline.invoke("программисты", config)
    print(f"  Финал: {result[:150]}")


if __name__ == "__main__":
    example_conditions_and_loops()


ПРИМЕР 2: Условная логика и циклы


  Черновик: Программист пошёл в магазин за хлебом, а вернулся с двумя багами и одним костылё...


  Итерация 1: APPROVED...
  Одобрено!
  Финал: Программист пошёл в магазин за хлебом, а вернулся с двумя багами и одним костылём.


In [6]:
# ============================================================================
# ЧАСТЬ 3: ПОБОЧНЫЕ ЭФФЕКТЫ (GOTCHA)
# ============================================================================
def example_side_effects_gotcha():
    """
    Демонстрация: код вне @task перевыполняется при resume.
    Побочные эффекты ОБЯЗАНЫ быть внутри @task.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 3: Побочные эффекты (gotcha)")
    print("=" * 60)

    call_log = []

    @task
    def safe_api_call(query: str) -> str:
        """Этот вызов НЕ повторится при resume — результат из checkpoint."""
        call_log.append(f"safe: {query}")
        return f"Result for {query}"

    @entrypoint(checkpointer=InMemorySaver())
    def pipeline(query: str) -> str:
        # ⚠️ Этот код ПЕРЕВЫПОЛНИТСЯ при resume!
        call_log.append(f"outside_task: {query}")

        # ✅ Этот вызов НЕ повторится — @task с checkpointing
        result = safe_api_call(query).result()

        # Прерывание для демонстрации
        approved = interrupt({"result": result, "question": "OK?"})

        return f"{result} (approved={approved})"

    config = {"configurable": {"thread_id": "gotcha-1"}}

    # Первый вызов — останавливается на interrupt
    call_log.clear()
    pipeline.invoke("test", config)
    print(f"  После первого вызова: {call_log}")

    # Resume — entrypoint перевыполняется, но @task берётся из checkpoint
    call_log.clear()
    result = pipeline.invoke(Command(resume=True), config)
    print(f"  После resume: {call_log}")
    print(f"  Обратите внимание: 'outside_task' выполнился снова, 'safe' — нет")
    print(f"  Результат: {result}")


if __name__ == "__main__":
    example_side_effects_gotcha()


ПРИМЕР 3: Побочные эффекты (gotcha)
  После первого вызова: ['outside_task: test', 'safe: test']
  После resume: ['outside_task: test']
  Обратите внимание: 'outside_task' выполнился снова, 'safe' — нет
  Результат: Result for test (approved=True)


In [7]:
# ============================================================================
# ЧАСТЬ 4: RETRY И КЭШИРОВАНИЕ НА УРОВНЕ ЗАДАЧИ
# ============================================================================
def example_retry_and_cache():
    """
    @task с retry_policy и cache_policy.
    Retry — автоматическое повторение при ошибке.
    Cache — кэширование результата по аргументам.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 4: Retry и кэширование")
    print("=" * 60)

    call_counter = {"fetch": 0, "compute": 0}

    @task(
        retry_policy=RetryPolicy(
            max_attempts=3,
            initial_interval=0.3,
            backoff_factor=2.0,
        )
    )
    def fetch_data(url: str) -> str:
        """Задача с retry — повторяет при ошибке."""
        call_counter["fetch"] += 1
        attempt = call_counter["fetch"]
        print(f"  [fetch] Попытка #{attempt}")
        if attempt < 2:
            raise ConnectionError("Network timeout")
        return f"Data from {url}"

    @task(cache_policy=CachePolicy(ttl=60))
    def compute(data: str) -> str:
        """Задача с кэшем — повторный вызов с теми же аргументами берётся из кэша."""
        call_counter["compute"] += 1
        print(f"  [compute] Вычисление #{call_counter['compute']}")
        return f"Processed: {data}"

    @entrypoint(checkpointer=InMemorySaver())
    def pipeline(url: str) -> str:
        data = fetch_data(url).result()
        result = compute(data).result()
        return result

    config = {"configurable": {"thread_id": "retry-cache-1"}}
    result = pipeline.invoke("https://api.example.com/data", config)
    print(f"  Результат: {result}")


if __name__ == "__main__":
    example_retry_and_cache()


ПРИМЕР 4: Retry и кэширование
  [fetch] Попытка #1


  [fetch] Попытка #2
  [compute] Вычисление #1
  Результат: Processed: Data from https://api.example.com/data


In [8]:
# ============================================================================
# ЧАСТЬ 5: ПАРАЛЛЕЛЬНОЕ ВЫПОЛНЕНИЕ ЗАДАЧ
# ============================================================================
def example_parallel_tasks():
    """
    Параллельный запуск задач через list comprehension + .result().
    Аналог Send, но в функциональном стиле.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 5: Параллельное выполнение задач")
    print("=" * 60)

    llm = get_llm()

    @task
    def analyze_topic(topic: str) -> str:
        msg = llm.invoke(f"Одним предложением опиши: {topic}")
        return msg.content

    @task
    def synthesize(analyses: list[str]) -> str:
        combined = "\n".join(f"- {a}" for a in analyses)
        msg = llm.invoke(f"Объедини эти факты в один абзац:\n{combined}")
        return msg.content

    @entrypoint(checkpointer=InMemorySaver())
    def research_pipeline(topics: list[str]) -> str:
        # Запуск задач параллельно
        futures = [analyze_topic(t) for t in topics]
        # Сбор результатов
        analyses = [f.result() for f in futures]
        print(f"  Проанализировано {len(analyses)} тем")

        # Синтез
        summary = synthesize(analyses).result()
        return summary

    config = {"configurable": {"thread_id": "parallel-1"}}
    result = research_pipeline.invoke(
        ["Нейросети", "Квантовые компьютеры", "Робототехника"], config
    )
    print(f"  Синтез: {result[:200]}...")


if __name__ == "__main__":
    example_parallel_tasks()


ПРИМЕР 5: Параллельное выполнение задач


  Проанализировано 3 тем


  Синтез: Нейросети — это алгоритмы, вдохновлённые работой человеческого мозга, которые обучаются на данных и умеют находить закономерности, распознавать образы и делать прогнозы; квантовые компьютеры — это уст...


In [9]:
# ============================================================================
# ЧАСТЬ 6: entrypoint.final — РАЗДЕЛЕНИЕ RETURN И SAVE
# ============================================================================
def example_entrypoint_final():
    """
    entrypoint.final(value=..., save=...) — возвращает одно значение
    вызывающему коду, а сохраняет другое в checkpoint для previous.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 6: entrypoint.final")
    print("=" * 60)

    @entrypoint(checkpointer=InMemorySaver())
    def accumulator(number: int, *, previous: int | None = None) -> entrypoint.final:
        previous = previous or 0
        total = previous + number
        print(f"  Вход: {number}, Previous: {previous}, Total: {total}")

        # value — что получит вызывающий код (только previous)
        # save — что сохранится для следующего вызова (total)
        return entrypoint.final(value=previous, save=total)

    config = {"configurable": {"thread_id": "accum-1"}}

    r1 = accumulator.invoke(10, config)
    print(f"  Вызов 1: вернул {r1} (previous=0, saved total=10)")

    r2 = accumulator.invoke(5, config)
    print(f"  Вызов 2: вернул {r2} (previous=10, saved total=15)")

    r3 = accumulator.invoke(3, config)
    print(f"  Вызов 3: вернул {r3} (previous=15, saved total=18)")


if __name__ == "__main__":
    example_entrypoint_final()


ПРИМЕР 6: entrypoint.final
  Вход: 10, Previous: 0, Total: 10
  Вызов 1: вернул 0 (previous=0, saved total=10)
  Вход: 5, Previous: 10, Total: 15
  Вызов 2: вернул 10 (previous=10, saved total=15)
  Вход: 3, Previous: 15, Total: 18
  Вызов 3: вернул 15 (previous=15, saved total=18)


In [10]:
# ============================================================================
# ЧАСТЬ 7: PREVIOUS — КОРОТКОСРОЧНАЯ ПАМЯТЬ
# ============================================================================
def example_previous_memory():
    """
    previous — доступ к результату предыдущего вызова на том же thread.
    Простейшая форма памяти между вызовами.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 7: previous — короткосрочная память")
    print("=" * 60)

    @entrypoint(checkpointer=InMemorySaver())
    def chatbot(message: str, *, previous: list[str] | None = None) -> list[str]:
        history = previous or []
        history.append(f"User: {message}")

        llm = get_llm()
        context = "\n".join(history[-6:])  # Последние 6 сообщений
        msg = llm.invoke(f"История:\n{context}\nОтветь кратко.")
        history.append(f"Bot: {msg.content}")

        return history

    config = {"configurable": {"thread_id": "chat-1"}}

    h1 = chatbot.invoke("Привет! Меня зовут Алексей.", config)
    print(f"  Сообщений: {len(h1)}")

    h2 = chatbot.invoke("Как меня зовут?", config)
    print(f"  Сообщений: {len(h2)}")
    print(f"  Последний ответ: {h2[-1][:100]}...")


if __name__ == "__main__":
    example_previous_memory()


ПРИМЕР 7: previous — короткосрочная память


  Сообщений: 2


  Сообщений: 4
  Последний ответ: Bot: Алексей....


In [11]:
# ============================================================================
# ЧАСТЬ 8: INTERRUPT В FUNCTIONAL API
# ============================================================================
def example_interrupt_in_functional():
    """
    interrupt() работает так же, как в StateGraph.
    Останавливает выполнение, ждёт Command(resume=...).
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 8: interrupt() в Functional API")
    print("=" * 60)

    llm = get_llm()

    @task
    def write_email(topic: str) -> str:
        msg = llm.invoke(f"Напиши короткое деловое письмо на тему: {topic}")
        return msg.content

    @task
    def send_email(text: str) -> str:
        return f"Email отправлен: {text[:50]}..."

    @entrypoint(checkpointer=InMemorySaver())
    def email_pipeline(topic: str) -> str:
        draft = write_email(topic).result()

        # Пауза — ждём одобрения человека
        decision = interrupt({
            "draft": draft[:200],
            "question": "Отправить? (approve / reject)",
        })

        if decision.get("action") == "approve":
            result = send_email(draft).result()
            return result
        else:
            return "Email отклонён пользователем"

    config = {"configurable": {"thread_id": "email-1"}}

    # Шаг 1: запуск — остановится на interrupt
    print("  Шаг 1: Генерация черновика...")
    for chunk in email_pipeline.stream(
        "Приглашение на встречу", config, stream_mode="updates"
    ):
        if "__interrupt__" in str(chunk):
            print("  Граф остановлен — ждёт одобрения")

    # Шаг 2: одобрение
    print("  Шаг 2: Одобрение...")
    result = email_pipeline.invoke(
        Command(resume={"action": "approve"}), config
    )
    print(f"  Результат: {result}")


if __name__ == "__main__":
    example_interrupt_in_functional()


ПРИМЕР 8: interrupt() в Functional API
  Шаг 1: Генерация черновика...


  Граф остановлен — ждёт одобрения
  Шаг 2: Одобрение...
  Результат: Email отправлен: Конечно. Вот короткий деловой вариант:

**Тема:** ...


In [12]:
# ============================================================================
# ЧАСТЬ 9: КАСТОМНЫЙ СТРИМИНГ
# ============================================================================
def example_custom_streaming():
    """
    get_stream_writer() — отправка кастомных данных в stream.
    Удобно для индикации прогресса.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 9: Кастомный стриминг")
    print("=" * 60)

    from langgraph.config import get_stream_writer

    @task
    def step_one(x: int) -> int:
        return x * 2

    @task
    def step_two(x: int) -> int:
        return x + 10

    @entrypoint(checkpointer=InMemorySaver())
    def pipeline(x: int) -> int:
        writer = get_stream_writer()

        writer({"status": "starting", "input": x})
        r1 = step_one(x).result()
        writer({"status": "step_one_done", "intermediate": r1})
        r2 = step_two(r1).result()
        writer({"status": "complete", "result": r2})

        return r2

    config = {"configurable": {"thread_id": "stream-1"}}

    print("  Стриминг:")
    for mode, chunk in pipeline.stream(
        5, config, stream_mode=["custom", "updates"]
    ):
        if mode == "custom":
            print(f"    [custom] {chunk}")
        elif mode == "updates":
            print(f"    [update] {chunk}")


if __name__ == "__main__":
    example_custom_streaming()


ПРИМЕР 9: Кастомный стриминг
  Стриминг:
    [custom] {'status': 'starting', 'input': 5}
    [update] {'step_one': 10}
    [custom] {'status': 'step_one_done', 'intermediate': 10}
    [update] {'step_two': 20}
    [custom] {'status': 'complete', 'result': 20}
    [update] {'pipeline': 20}
